In [3]:
from statsmodels.stats import inter_rater as irr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.metrics import balanced_accuracy_score, matthews_corrcoef, cohen_kappa_score, roc_auc_score, log_loss


In [4]:
df = pd.read_csv('df_final.csv')
df.head()

,Unnamed: 0,index,post_id,source_platform,post_text,token_count,majority_label,normal_votes,offensive_votes,hatespeech_votes,annotator_1_label,annotator_1_targets_text,annotator_2_label,annotator_2_targets_text,annotator_3_label,annotator_3_targets_text,rationale_count,post_tokens,annotator_count,rationales,labels,annotator_1_id,annotator_1_targets,annotator_2_id,annotator_2_targets,annotator_3_id,annotator_3_targets,rationale_1,rationale_2,rationale_3,majority
0,0,0,10001291_gab,gab,jews are responsible for destroying my towns c...,22,hatespeech,0,0,3,hatespeech,Jewish,hatespeech,Jewish,hatespeech,Jewish,2,"['jews', 'are', 'responsible', 'for', 'destroy...",3,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","['hatespeech', 'hatespeech', 'hatespeech']",199,['Jewish'],200,['Jewish'],201,['Jewish'],"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",NaN,['Jewish']
1,1,1,10002346_gab,gab,leftist mind control fails i think this overst...,33,normal,2,1,0,normal,"Men, Other",offensive,Other,normal,Women,0,"['leftist', 'mind', 'control', 'fails', 'i', '...",3,[],"['normal', 'offensive', 'normal']",202,"['Men', 'Other']",203,['Other'],204,['Women'],NaN,NaN,NaN,['Other']
2,2,3,10008196_gab,gab,it why they behave like animals no matter wher...,52,hatespeech,1,0,2,hatespeech,African,hatespeech,African,normal,NaN,2,"['it', 'why', 'they', 'behave', 'like', 'anima...",3,"[[0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","['hatespeech', 'hatespeech', 'normal']",200,['African'],208,['African'],209,['None'],"[0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",NaN,['African']
3,3,4,10009556_gab,gab,why are people so offended by the word paki su...,19,offensive,0,3,0,offensive,Asian,offensive,Arab,offensive,Arab,3,"['why', 'are', 'people', 'so', 'offended', 'by...",3,"[[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,...","['offensive', 'offensive', 'offensive']",27,['Asian'],1,['Arab'],52,['Arab'],"[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...",['Arab']
4,4,5,10010219_gab,gab,sugar grove that out <number> sugar grove is n...,41,offensive,0,3,0,offensive,Caucasian,offensive,African,offensive,Economic,3,"['sugar', 'grove', 'that', 'out', '<number>', ...",3,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,...","['offensive', 'offensive', 'offensive']",27,['Caucasian'],41,['African'],62,['Economic'],"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[]


In [5]:
# Trying to answer: How accurate are humans? What is the inter-rater agreement of different raters?
# Making list of lists to observe differing ratings
ann1_rating = df['annotator_1_targets']
ann2_rating = df['annotator_2_targets']
ann3_rating = df['annotator_3_targets']
data = {'rater1': ann1_rating, 'rater2': ann2_rating, 'rater3': ann3_rating}

raters = pd.DataFrame(data)
raters

,rater1,rater2,rater3
0,['Jewish'],['Jewish'],['Jewish']
1,"['Men', 'Other']",['Other'],['Women']
2,['African'],['African'],['None']
3,['Asian'],['Arab'],['Arab']
4,['Caucasian'],['African'],['Economic']
...,...,...,...
19224,['Islam'],['Other'],['None']
19225,['Hispanic'],['Asian'],['Asian']
19226,['Islam'],['Islam'],['Islam']
19227,"['Minority', 'Refugee', 'Indian']",['Indian'],"['Refugee', 'Islam']"


In [ ]:
# Calculating Fleiss' kappa over Cohen's kappa due to ability to compare >2 raters
# Fleiss' kappa uses a single pooled marginal (Pij), treating all raters as if they draw from the same distribution. 
# Averages away noise
# First, converting data to format accepted by Fleiss' kappa
table, categories = irr.aggregate_raters(raters)

# Now calculating the kappa
f_kappa = irr.fleiss_kappa(table, method = 'fleiss')
rand_kappa = irr.fleiss_kappa(table, method = 'rand')

print(f_kappa, rand_kappa)

0.46425474930691374 0.5469024941339379
